# 🚀 PELATIHAN OTOMATIS YOLO11 (DILENGKAPI FITUR ANTI-GAGAL / AUTO-RESUME)
### Program Tesis S2 — Multi-Dataset Roboflow & YOLO11 on GPU

Notebook ini dilengkapi **Checkpoint Otomatis ke Google Drive**. Jika koneksi internet Anda terputus di tengah jalan, Anda **TIDAK PERLU mengulang dari awal** — cukup jalankan sel pelatihan lagi, dan sistem akan **otomatis melanjutkan dari epoch terakhir!**

**Langkah Penggunaan:**
1. Pastikan Runtime GPU aktif: **Runtime → Change runtime type → T4 GPU**.
2. Jalankan semua sel: **Runtime → Run all** (`Ctrl + F9`).
3. Izinkan akses Google Drive saat diminta.
4. Santai dan tunggu hingga selesai — model terbaik (`best.pt`) tersimpan permanen di Drive!

In [ ]:
# @title 1. Pemasangan Dependensi & Library Machine Learning Terbaru
!pip install -q ultralytics roboflow pyyaml
print('✅ [1/5] Library Ultralytics YOLO11 & Roboflow berhasil dipasang!')

In [ ]:
# @title 2. Menghubungkan Google Drive & Menyiapkan Folder Checkpoint Aman
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Folder penyimpanan permanen di Google Drive Anda
DRIVE_DIR = Path('/content/drive/MyDrive/Program Tesis Colab')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = DRIVE_DIR / 'yolo11_training_checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ [2/5] Google Drive terhubung!')
print(f'   📁 Folder Checkpoint: {CHECKPOINT_DIR}')
print(f'   📁 Folder Model Final: {DRIVE_DIR}')

In [ ]:
# @title 3. Mengunduh Seluruh Dataset dari Roboflow
import os
from roboflow import Roboflow

API_KEY = "OR8FgoTeEOMfERN1tfMU"
rf = Roboflow(api_key=API_KEY)

print('📥 Mengunduh Dataset 1: UNUSIDA (cars-crash-detection v14)...')
ds1 = rf.workspace("universitas-nahdlatul-ulama-sidoarjo").project("cars-crash-detection").version(14).download("yolov11")

print('📥 Mengunduh Dataset 2: Traffic Accident 4w4pe...')
try:
    ds2 = rf.workspace("learnyolo-qaxxo").project("traffic-accident-4w4pe").version(1).download("yolov11")
except Exception as e:
    print(f'⚠️ Info ds2: {e}')
    ds2 = None

print('📥 Mengunduh Dataset 3: Accident Detection oyj8z...')
try:
    ds3 = rf.workspace("laiba-abid").project("accident-detection-oyj8z").version(2).download("yolov11")
except Exception as e:
    print(f'⚠️ Info ds3: {e}')
    ds3 = None

print('✅ [3/5] Seluruh dataset berhasil diunduh ke lingkungan Colab!')

In [ ]:
# @title 4. Menggabungkan & Menyeimbangkan Dataset (Unified Dataset Pipeline)
import shutil, yaml
from pathlib import Path

UNIFIED_DIR = Path('/content/unified_accident_dataset')
if UNIFIED_DIR.exists():
    shutil.rmtree(UNIFIED_DIR)

for split in ['train', 'valid', 'test']:
    (UNIFIED_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (UNIFIED_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# Baca konfigurasi kelas dari dataset utama UNUSIDA
main_yaml = Path(ds1.location) / 'data.yaml'
with open(main_yaml, 'r') as f:
    cfg = yaml.safe_load(f)

UNIFIED_NAMES = cfg.get('names', ['multiple_accident', 'normal', 'single_accident'])
print('🎯 Daftar Kelas Utama:', UNIFIED_NAMES)

datasets = [('unusida', ds1)]
if ds2: datasets.append(('learnyolo', ds2))
if ds3: datasets.append(('laiba', ds3))

total_images = 0
for prefix, ds in datasets:
    ds_path = Path(ds.location)
    for split in ['train', 'valid', 'test']:
        img_dir = ds_path / split / 'images'
        lbl_dir = ds_path / split / 'labels'
        if not img_dir.exists():
            continue
        for img in img_dir.glob('*.*'):
            if img.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue
            new_name = f"{prefix}_{img.name}"
            shutil.copy(img, UNIFIED_DIR / split / 'images' / new_name)
            
            lbl_file = lbl_dir / f"{img.stem}.txt"
            if lbl_file.exists():
                shutil.copy(lbl_file, UNIFIED_DIR / split / 'labels' / f"{prefix}_{img.stem}.txt")
            total_images += 1

# Tulis data.yaml gabungan baru
unified_yaml = {
    'path': str(UNIFIED_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': UNIFIED_NAMES
}

with open(UNIFIED_DIR / 'data.yaml', 'w') as f:
    yaml.dump(unified_yaml, f)

print(f'✅ [4/5] Berhasil menggabungkan {total_images} citra ke: {UNIFIED_DIR / "data.yaml"}')

In [ ]:
# @title 5. Melatih Model YOLO11 (Didukung Auto-Resume ke Google Drive)
from ultralytics import YOLO
import shutil
from pathlib import Path

checkpoint_last = Path('/content/drive/MyDrive/Program Tesis Colab/yolo11_training_checkpoints/weights/last.pt')

if checkpoint_last.exists():
    print('🔄 DITEMUKAN CHECKPOINT SEBELUMNYA DI GOOGLE DRIVE!')
    print('👉 Melanjutkan pelatihan otomatis dari epoch terakhir...')
    model = YOLO(str(checkpoint_last))
    results = model.train(resume=True)
else:
    print('🚀 Memulai Pelatihan Baru Model YOLO11 Medium (yolo11m.pt)...')
    model = YOLO('yolo11m.pt')
    results = model.train(
        data='/content/unified_accident_dataset/data.yaml',
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,          # Berhenti otomatis jika akurasi sudah mencapai puncak maksimal
        device=0,             # GPU T4 Cores
        project='/content/drive/MyDrive/Program Tesis Colab',
        name='yolo11_training_checkpoints',
        exist_ok=True,
        save=True,
        plots=True,
        workers=4
    )

# Salin model final terbaik ke best.pt di Google Drive
best_src = Path('/content/drive/MyDrive/Program Tesis Colab/yolo11_training_checkpoints/weights/best.pt')
final_dest = Path('/content/drive/MyDrive/Program Tesis Colab/best.pt')

if best_src.exists():
    shutil.copy(best_src, final_dest)
    print('=' * 75)
    print('🎉 PELATIHAN SELESAI & SUKSES BESAR!')
    print(f'💾 Model Final YOLO11 Tersimpan Permanen di:')
    print(f'   👉 {final_dest}')
    print('=' * 75)
else:
    print('⚠️ Periksa folder checkpoint di Google Drive.')

In [ ]:
# @title 6. Menampilkan Evaluasi Hasil Pelatihan (Confusion Matrix & mAP Curve)
from IPython.display import Image, display
from pathlib import Path

run_dir = Path('/content/drive/MyDrive/Program Tesis Colab/yolo11_training_checkpoints')

for plot_name in ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png']:
    p = run_dir / plot_name
    if p.exists():
        print(f'📊 Menampilkan {plot_name}:')
        display(Image(filename=str(p)))
